# SQL Business Analysis

In [3]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

user = os.getenv("MYSQL_USER")
password = os.getenv("MYSQL_PASSWORD")
host = os.getenv("MYSQL_HOST")
database = os.getenv("MYSQL_DATABASE")

engine = create_engine(
    f"mysql+mysqlconnector://{user}:{password}@{host}/{database}"
)

## SALES ANALYSIS

#### TOTAL REVENUE

##### OBJECTIVE
Determine the total revenue generated in the dataset

In [ ]:
query = """ SELECT SUM(PAYMENT_VALUE) AS TOTAL_REVENUE 
          FROM PAYMENTS; """

df=pd.read_sql(query,engine)


df["TOTAL_REVENUE"]=df["TOTAL_REVENUE"].round(2)
print(df["TOTAL_REVENUE"])

#### INSIGHT

The company generated approximately **R$16.01 million** in total revenue.

This provides a baseline for evaluating sales performance across products, customers, and time periods.

## TOTAL ORDERS

### OBJECTIVE
Determine the total orders placed in the dataset.

In [46]:
query = """SELECT COUNT(*) AS TOTAL_ORDERS 
            FROM ORDERS"""

df=pd.read_sql(query,engine)


### INSIGHT

The dataset contains **99,441 orders**. This provides the baseline for understanding the scale of business operations and will be referenced in subsequent analyses.

## AVERAGE ORDER VALUE

### OBJECTIVE
To determine the average order value from the dataset.

In [47]:
query="""SELECT AVG(ORDER_TOTAL) AS AVG_ORDER_TOTAL FROM (SELECT ORDER_ID, SUM(PRICE)AS ORDER_TOTAL
                                                  FROM ORDER_ITEMS GROUP BY ORDER_ID) AS TEMP"""

df=pd.read_sql(query,engine)
df["AVG_ORDER_TOTAL"]=df["AVG_ORDER_TOTAL"].round(2)

### INSIGHT
The Average Order Value (AOV) represents the average amount customers spend on each purchase.

This metric helps evaluate customer purchasing behavior and serves as an important KPI for measuring sales performance. A higher AOV generally indicates that customers are purchasing more expensive products or buying multiple products per order.

## MONTHLY REVENUE TREND

### OBJECTIVE
Analyze how revenue changes over time.

In [48]:
query="""SELECT DATE_FORMAT(O.ORDER_PURCHASE_TIMESTAMP, "%Y-%m") AS MONTH, SUM(P.PAYMENT_VALUE) AS MONTHLY_REVENUE
FROM ORDERS AS O
JOIN PAYMENTS AS P
ON O.ORDER_ID=P.ORDER_ID
GROUP BY MONTH
ORDER BY MONTH"""

df=pd.read_sql(query,engine)



### INSIGHT
Identifies seasonal sales patterns, peak months, and periods of lower revenue.

## MONTHLY ORDER TREND

### OBJECTIVE
Analyze how orders changes over months.

In [49]:
query="""SELECT DATE_FORMAT(ORDER_PURCHASE_TIMESTAMP, '%Y-%m') AS MONTH,COUNT(*) AS TOTAL_ORDERS
FROM ORDERS 
GROUP BY MONTH
ORDER BY MONTH;"""

df=pd.read_sql(query,engine)

### INSGIHT
Identifies seasonal sales patterns, peak months, and periods of lower orders receiving rate.

# CUSTOMERS ANALYSIS

## TOP 10 CUSTOMERS BY SPENDING

### OBJECTIVE 
Identify customers contributing the highest total revenue

In [50]:
query="""SELECT C.CUSTOMER_ID, COUNT(DISTINCT O.ORDER_ID) AS NUMBER_OF_ORDERS, SUM(P.PAYMENT_VALUE) AS TOTAL_SPENDING
FROM PAYMENTS AS P
JOIN ORDERS AS O 
ON P.ORDER_ID=O.ORDER_ID
JOIN CUSTOMERS AS C
ON C.CUSTOMER_ID=O.CUSTOMER_ID
GROUP BY C.CUSTOMER_ID
ORDER BY TOTAL_SPENDING DESC
LIMIT 10"""

df=pd.read_sql(query,engine)

### INSIGHT
Highlights high-value customers who generate the largest share of sales.

## CUSTOMERS WITH HIGHEST NUMBER OF ORDERS

### OBJECTIVE
Find customers who place orders most frequently.

In [51]:
query="""SELECT C.CUSTOMER_ID,COUNT(O.ORDER_ID) AS NUMBER_OF_ORDERS
FROM CUSTOMERS AS C 
JOIN ORDERS AS O
ON C.CUSTOMER_ID=O.CUSTOMER_ID
GROUP BY C.CUSTOMER_ID
ORDER BY NUMBER_OF_ORDERS DESC"""

df=pd.read_sql(query,engine)

### INSIGHT
Helps identify loyal and repeat customers.

## AVERAGE SPENDING PER CUSTOMER

### OBJECTIVE
Calculate the average spending for each customer.

In [52]:
query="""SELECT C.CUSTOMER_ID, AVG(P.PAYMENT_VALUE) AS AVG_SPENDING
FROM ORDERS AS O
JOIN PAYMENTS AS P
ON O.ORDER_ID=P.ORDER_ID
JOIN CUSTOMERS AS C
ON C.CUSTOMER_ID=O.CUSTOMER_ID
GROUP BY C.CUSTOMER_ID
ORDER BY AVG_SPENDING DESC"""

df=pd.read_sql(query,engine)

### INSIGHT
Reveals customers with consistently higher purchase values.

## REPEAT CUSTOMERS

### OBJECTIVE
Identify customers who placed more than one order.

In [53]:
query="""SELECT C.CUSTOMER_ID,COUNT(O.CUSTOMER_ID)-1 AS TIMES_REPEATED
FROM CUSTOMERS AS C 
JOIN ORDERS AS O
ON C.CUSTOMER_ID=O.CUSTOMER_ID
GROUP BY C.CUSTOMER_ID
HAVING TIMES_REPEATED>0
ORDER BY TIMES_REPEATED DESC"""

df=pd.read_sql(query,engine)

### INSIGHT
Measures customer retention and repeat purchasing behaviour.

## CUSTOMER DISTRIBUTION BY STATE

### OBJECTIVE
Count customers in each state.

In [54]:
query="""SELECT CUSTOMER_STATE,COUNT(*) AS NUMBER_OF_CUSTOMERS
FROM CUSTOMERS
GROUP BY CUSTOMER_STATE"""
df=pd.read_sql(query,engine)

### INSIGHT
Shows geographical distribution of the customer base.

# PRODUCT ANALYSIS

## TOP SELLING PODUCTS

### OBJECTIVE
Find the products purchased most frequently.

In [55]:
query="""SELECT PR.PRODUCT_ID,PR.PRODUCT_CATEGORY_NAME,COUNT(OI.ORDER_ID) AS TIMES_ORDER_PURCHASED
FROM PRODUCTS AS PR
JOIN ORDER_ITEMS AS OI
ON PR.PRODUCT_ID=OI.PRODUCT_ID
GROUP BY PR.PRODUCT_ID,PR.PRODUCT_CATEGORY_NAME
ORDER BY TIMES_ORDER_PURCHASED DESC
LIMIT 10"""

df=pd.read_sql(query,engine)

### INSIGHT
Reveals customer preferences and best-selling products.

## HIGHEST REVENUE PRODUCTS

### OBJECTIVE
Identify products generating the highest revenue.

In [56]:
query="""SELECT PR.PRODUCT_ID,PR.PRODUCT_CATEGORY_NAME,SUM(OI.PRICE) AS REVENUE
FROM PRODUCTS AS PR
JOIN ORDER_ITEMS AS OI
ON PR.PRODUCT_ID=OI.PRODUCT_ID
GROUP BY PR.PRODUCT_ID,PR.PRODUCT_CATEGORY_NAME
ORDER BY REVENUE DESC
LIMIT 10"""

df=pd.read_sql(query,engine)

### INSIGHT
Shows which products contribute the most to total sales.

## TOP PRODUCT CATEGORIES

### OBJECTIVE
Determine the categories of products which generate the highest revenue.

In [57]:
query="""SELECT PR.PRODUCT_CATEGORY_NAME,SUM(OI.PRICE) AS REVENUE_FROM_CATEGORY
FROM PRODUCTS AS PR
JOIN ORDER_ITEMS AS OI
ON PR.PRODUCT_ID=OI.PRODUCT_ID
GROUP BY PR.PRODUCT_CATEGORY_NAME
ORDER BY REVENUE_FROM_CATEGORY DESC
LIMIT 10"""

df=pd.read_sql(query,engine)

### INSIGHT
Identifies the categories of the top selling products.

## CATEGORIES WITH HIGHEST REVENUE

### OBJECTIVE
Determine which product categories generate the highest revenue.

In [58]:
query="""SELECT PR.PRODUCT_CATEGORY_NAME,SUM(OI.PRICE) AS REVENUE_FROM_CATEGORY
FROM PRODUCTS AS PR
JOIN ORDER_ITEMS AS OI
ON PR.PRODUCT_ID=OI.PRODUCT_ID
GROUP BY PR.PRODUCT_CATEGORY_NAME
ORDER BY REVENUE_FROM_CATEGORY DESC
LIMIT 10"""

df=pd.read_sql(query,engine)

### INSIGHT
Identifies the strongest-performing product categories.

## AVERAGE PRODUCT PRICE BY CATEGORY

### OBJECTIVE
Calculate the average product price for each category.

In [59]:
query="""SELECT PR.PRODUCT_CATEGORY_NAME,AVG(OI.PRICE) AS AVG_PRICE_BY_CATEGORY
FROM PRODUCTS AS PR
JOIN ORDER_ITEMS AS OI
ON PR.PRODUCT_ID=OI.PRODUCT_ID
GROUP BY PR.PRODUCT_CATEGORY_NAME"""

df=pd.read_sql(query,engine)

### INSIGHT
Compares pricing across different product categories.

# SELLERS ANALYSIS

## TOP SELLERS BY REVENUE

### OBJECTIVE
Calculate total revenue generated by each seller.

In [60]:
query="""SELECT S.SELLER_ID,SUM(OI.PRICE) AS REVENUE_BY_SELLER
FROM SELLERS AS S
JOIN ORDER_ITEMS AS OI
ON S.SELLER_ID=OI.SELLER_ID
GROUP BY S.SELLER_ID
ORDER BY REVENUE_BY_SELLER DESC
LIMIT 10"""

df=pd.read_sql(query,engine)

### INSIGHT
Identifies the highest-performing sellers.

## TOP SELLERS BY ORDERS

### OBJECTIVE
Find sellers fulfilling the most orders.

In [61]:
query="""SELECT S.SELLER_ID,COUNT(DISTINCT OI.ORDER_ID) AS ORDERS_BY_SELLER
FROM SELLERS AS S
JOIN ORDER_ITEMS AS OI
ON S.SELLER_ID=OI.SELLER_ID
GROUP BY S.SELLER_ID
ORDER BY ORDERS_BY_SELLER DESC
LIMIT 10"""

df=pd.read_sql(query,engine)

### INSIGHT
Measures seller activity and order volume.

## AVERAGE SELLER REVENUE

### OBJECTIVE
Calculate the average total revenue generated by sellers.

In [62]:
query="""SELECT AVG(SELLER_REVENUE) AS AVG_SELLER_REVENUE 
FROM(SELECT SELLER_ID,SUM(PRICE) AS SELLER_REVENUE
	 FROM ORDER_ITEMS GROUP BY SELLER_ID)AS TEMP"""

df=pd.read_sql(query,engine)

### INSIGHT
Provides a benchmark for seller performance.

## SELLER DISTRIBUTION BY STATE

### OBJECTIVE
Count sellers in each state.

In [63]:
query="""SELECT SELLER_STATE,COUNT(SELLER_ID) AS NUMBER_OF_SELLERS
FROM SELLERS
GROUP BY SELLER_STATE
ORDER BY NUMBER_OF_SELLERS DESC"""

df=pd.read_sql(query,engine)

### INSIGHT
Shows seller distribution across regions.

# REVIEWS ANALYSIS

## AVERAGE REVIEW SCORE

### OBJECTIVE
To get the average of the reviews from the dataset.


In [64]:
query="""SELECT AVG(REVIEW_SCORE) FROM REVIEWS"""

df=pd.read_sql(query,engine)

### INSIGHT
To get the overall review score for all the orders

## REVIEW SCORE DISTRIBUTION

### OBJECTIVE
Analyze the frequency of each review score.

In [65]:
query="""SELECT REVIEW_SCORE,COUNT(*) AS REVIEW_COUNT
FROM REVIEWS
GROUP BY REVIEW_SCORE
ORDER BY REVIEW_SCORE"""

df=pd.read_sql(query,engine)

### INSIGHT
Measures customer satisfaction and overall review patterns.

## REVENUE VS REVIEW SCORE

### OBJECTIVE
Compare total revenue across different review scores.

In [66]:
query="""SELECT R.REVIEW_SCORE,SUM(P.PAYMENT_VALUE) AS REVENUE
FROM REVIEWS AS R
JOIN PAYMENTS AS P
ON R.ORDER_ID=P.ORDER_ID
GROUP BY R.REVIEW_SCORE
ORDER BY R.REVIEW_SCORE"""

df=pd.read_sql(query,engine)

### INSIGHT
Evaluates whether higher-rated orders contribute more revenue.

## CATEGORIES WITH LOWEST RATINGS

### OBJECTIVE
Calculate the average review score for each product category.

In [67]:
query="""SELECT PR.PRODUCT_CATEGORY_NAME,AVG(R.REVIEW_SCORE) AS AVG_REVIEW_SCORE
FROM PRODUCTS AS PR
JOIN ORDER_ITEMS AS OI
ON PR.PRODUCT_ID=OI.PRODUCT_ID
JOIN REVIEWS AS R 
ON R.ORDER_ID=OI.ORDER_ID
GROUP BY PR.PRODUCT_CATEGORY_NAME
ORDER BY AVG_REVIEW_SCORE
LIMIT 10"""

df=pd.read_sql(query,engine)

### INSIGHT
Identifies categories with the highest and lowest customer satisfaction.

# DELIVERY ANALYSIS

## AVERAGE DELIVERY TIME

### OBJECTIVE
Calculate the average number of days taken to deliver orders.

In [68]:
query="""SELECT AVG(DATEDIFF(ORDER_DELIVERED_CUSTOMER_DATE,ORDER_PURCHASE_TIMESTAMP))
AS AVG_DELIVERY_DAYS FROM ORDERS"""

df=pd.read_sql(query,engine)

### INSIGHT
Measures the overall efficiency of the delivery process.

## FASTEST STATES

### OBJECTIVE
Identify states with the shortest delivery times.

In [69]:
query="""SELECT C.CUSTOMER_STATE, AVG(DATEDIFF(O.ORDER_DELIVERED_CUSTOMER_DATE,O.ORDER_PURCHASE_TIMESTAMP))
AS NUMBER_OF_DELIVERY_DAYS 
FROM ORDERS AS O
JOIN CUSTOMERS AS C 
ON O.CUSTOMER_ID=C.CUSTOMER_ID
WHERE O.ORDER_DELIVERED_CUSTOMER_DATE IS NOT NULL
GROUP BY C.CUSTOMER_STATE
ORDER BY NUMBER_OF_DELIVERY_DAYS ASC
LIMIT 10"""

df=pd.read_sql(query,engine)

### INSIGHT
Highlights regions with the most efficient deliveries.

## SLOWEST STATES

### OBJECTIVE
Identify states with the longest delivery times.

In [70]:
query="""SELECT C.CUSTOMER_STATE, AVG(DATEDIFF(O.ORDER_DELIVERED_CUSTOMER_DATE,O.ORDER_PURCHASE_TIMESTAMP))
AS NUMBER_OF_DELIVERY_DAYS 
FROM ORDERS AS O
JOIN CUSTOMERS AS C 
ON O.CUSTOMER_ID=C.CUSTOMER_ID
WHERE O.ORDER_DELIVERED_CUSTOMER_DATE IS NOT NULL
GROUP BY C.CUSTOMER_STATE
ORDER BY NUMBER_OF_DELIVERY_DAYS DESC
LIMIT 10"""

df=pd.read_sql(query,engine)

### INSIGHT
Helps detect regions where delivery performance needs improvement.

## ORDERS DELIVERED LATE

### OBJECTIVE
Find orders delivered after the estimated delivery date.

In [71]:
query="""SELECT O.ORDER_ID FROM ORDERS AS O
WHERE O.ORDER_DELIVERED_CUSTOMER_DATE > O.ORDER_ESTIMATED_DELIVERY_DATE"""

df=pd.read_sql(query,engine)

### INSIGHT
Measures delayed deliveries and potential customer dissatisfaction.

## DELIVERY DELAY DISTRIBUTION

### OBJECTIVE
Analyze how delivery delays are distributed across all orders.

In [72]:
query="""SELECT DATEDIFF(O.ORDER_DELIVERED_CUSTOMER_DATE,O.ORDER_ESTIMATED_DELIVERY_DATE) AS DELAY_DAYS,
COUNT(*) AS NUMBER_OF_ORDERS
FROM ORDERS AS O
WHERE O.ORDER_DELIVERED_CUSTOMER_DATE IS NOT NULL
GROUP BY DELAY_DAYS
ORDER BY DELAY_DAYS DESC"""

df=pd.read_sql(query,engine)

### INSIGHT
Shows whether most orders are delivered early, on time, or late, and identifies the frequency of delivery delays.